In [1]:
import torch
import os, sys

import polars as pl

sys.path.insert(0, os.path.abspath(".."))

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/paper_research/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/paper_research/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

True
1
12.8
2.9.0.dev20250716+cu128
NVIDIA GeForce RTX 5080
2.9.0.dev20250716+cu128


In [2]:
marked_df = pl.read_parquet(DIR + 'sample_data/marked_target_df_k_4.parquet')
marked_df = (
    marked_df
    .sort(["oper_part_no", "seq"])
    .with_columns(pl.cum_count('oper_part_no').over("oper_part_no").alias("_rn"))
    .filter(pl.col("_rn") > 1)        # 첫 이벤트 제거
    .drop("_rn")
)
print(f"K Num: {int(marked_df.select((pl.col("mark").max() + 1).alias("k")).item())}")
marked_df

K Num: 4


oper_part_no,demand_dt,seq,delta_t,demand_qty,z,mark
str,i64,u32,i32,f64,f64,i32
"""0001-1002""",201902,44,43,1.0,0.693147,0
"""0001-1002""",202309,260,216,1.0,0.693147,0
"""0001-1002""",202312,263,3,1.0,0.693147,0
"""0001-1002""",202609,416,153,1.0,0.693147,0
"""0001-1002""",202612,419,3,1.0,0.693147,0
…,…,…,…,…,…,…
"""ZZ90239""",202010,6,5,1.0,0.693147,0
"""ZZ90239""",202111,60,54,1.0,0.693147,0
"""ZZ90239""",202306,159,99,1.0,0.693147,0


In [3]:
(
    marked_df.group_by("mark").len()
    .sort("len", descending=True)
    .with_columns((pl.col("len") / pl.col("len").sum()).alias("ratio"))
)

mark,len,ratio
i32,u32,f64
0,134917,0.614653
1,82272,0.374814
2,2084,0.009494
3,228,0.001039


In [4]:
from torch.utils.data import DataLoader
from data_loader.event_seq_data_module import RMTPPDataset, time_split_events, collate_next_event, \
    RMTPPWeekLookbackDataset, collate_week_lookback
from models.RMTPPs.RMTPP import RMTPPConfig
from utils.training import TrainingConfig, train_rmtpp

training_config = TrainingConfig(
    lookback = 3,
    batch_size = 256,
    max_seq_len = 64,
    lr = 1e-3,
    epochs = 30,
    val_ratio = 0.2,
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
    lambda_dt = 1.0,
)
train_events, val_events = time_split_events(marked_df, val_ratio = training_config.val_ratio)

train_ds = RMTPPDataset(train_events, lookback = training_config.lookback)
val_ds = RMTPPDataset(val_events, lookback = training_config.lookback)

train_loader = DataLoader(train_ds, batch_size = training_config.batch_size, shuffle = True, num_workers = 0, collate_fn = collate_next_event, drop_last = True)
val_loader = DataLoader(val_ds, batch_size = training_config.batch_size, shuffle = False, num_workers = 0, collate_fn = collate_next_event)
print("event concept: train_events rows:", train_events.height)
print("event concept: val_events rows  :", val_events.height)
print("event concept: train_ds len:", len(train_ds))
print("event concept: val_ds len  :", len(val_ds))


training_config = TrainingConfig(
    lookback = 52,
    batch_size = 256,
    max_seq_len = 64,
    lr = 1e-3,
    epochs = 30,
    val_ratio = 0.2,
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
    lambda_dt = 1.0,
)
train_ds = RMTPPWeekLookbackDataset(
        marked_df,
        lookback_weeks=training_config.lookback,  # 새 파라미터
        max_seq_len=training_config.max_seq_len,
        val_ratio=training_config.val_ratio,
        mode="train",
    )
val_ds = RMTPPWeekLookbackDataset(
    marked_df,
    lookback_weeks=training_config.lookback,
    max_seq_len=training_config.max_seq_len,
    val_ratio=training_config.val_ratio,
    mode="val",
)

train_loader = DataLoader(train_ds, batch_size = training_config.batch_size, shuffle=True, collate_fn = collate_week_lookback, drop_last=True)
val_loader = DataLoader(val_ds, batch_size = training_config.batch_size, shuffle=False, collate_fn = collate_week_lookback)

print("ts_concept: train_events rows:", train_events.height)
print("ts_concept: val_events rows  :", val_events.height)
print("ts_concept: train_ds len:", len(train_ds))
print("ts_concept: val_ds len  :", len(val_ds))

train_events

event concept: train_events rows: 145682
event concept: val_events rows  : 73819
event concept: train_ds len: 73840
event concept: val_ds len  : 11643
ts_concept: train_events rows: 145682
ts_concept: val_events rows  : 73819
ts_concept: train_ds len: 145682
ts_concept: val_ds len  : 50432


oper_part_no,demand_dt,seq,delta_t,demand_qty,z,mark
str,i64,u32,i32,f64,f64,i32
"""0001-1002""",201902,44,43,1.0,0.693147,0
"""0001-1002""",202309,260,216,1.0,0.693147,0
"""0001-1002""",202312,263,3,1.0,0.693147,0
"""0011-2-1-04""",202102,157,156,10.0,2.397895,1
"""0011-2-1-04""",202104,159,2,10.0,2.397895,1
…,…,…,…,…,…,…
"""ZZ90237""",202205,162,4,3.0,1.386294,1
"""ZZ90237""",202207,164,2,1.0,0.693147,0
"""ZZ90239""",202010,6,5,1.0,0.693147,0


In [5]:
from models.RMTPPs.RMTPP import RMTPPConfig
from utils.training import TrainingConfig, train_rmtpp


K = marked_df.select(pl.col('mark').max() + 1).item()

training_config = TrainingConfig(
    lookback = 52,
    max_seq_len = 64,
    batch_size = 256,
    lr = 3e-4,  # 1e-4
    epochs = 30,
    val_ratio = 0.2,
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
    lambda_dt = 1.0,
    grad_clip = 1.0
)

rmtpp_config = RMTPPConfig(
    num_marks = K + 1,
    mark_emb_dim = 32,
    rnn_hidden_dim = 256,
    rnn_type = 'gru',
    dropout = 0,
    eps = 1e-8,
    w_min = 5e-3,
    exp_clamp = 150.00
)

model, info = train_rmtpp(
    marked_df,
    rmtpp_config = rmtpp_config,
    training_config = training_config,
)

model.eval().to(training_config.device)
info

[Epoch 01] train_loss=27.90199419 | val_acc=0.75091212 val_dt_mae=33.25147403 | val_dt_rmse=43.40307672 | val_nll_time=6.705825 val_nll_marker=0.584283 val_nll=7.290108 | total=50432 | correct=37870 | steps=194356
[Epoch 02] train_loss=5.81317825 | val_acc=0.78705980 val_dt_mae=30.29750556 | val_dt_rmse=41.46250506 | val_nll_time=5.601249 val_nll_marker=0.568611 val_nll=6.169859 | total=50432 | correct=39693 | steps=194356
[Epoch 03] train_loss=5.30178939 | val_acc=0.78777364 val_dt_mae=29.54240619 | val_dt_rmse=41.84555321 | val_nll_time=4.854328 val_nll_marker=0.560720 val_nll=5.415049 | total=50432 | correct=39729 | steps=194356
[Epoch 04] train_loss=4.95149752 | val_acc=0.78973668 val_dt_mae=32.21316576 | val_dt_rmse=43.02901045 | val_nll_time=4.046458 val_nll_marker=0.558267 val_nll=4.604725 | total=50432 | correct=39828 | steps=194356
[Epoch 05] train_loss=4.72745547 | val_acc=0.79066862 val_dt_mae=28.42642873 | val_dt_rmse=42.51968333 | val_nll_time=4.615158 val_nll_marker=0.556

{'best_score': 0.5321502763002657}

In [6]:
print(marked_df.select(pl.col("delta_t")).describe())
# marked_df.select(pl.col('delta_t')).with_columns([
#     pl.col('delta_t').quantile(0.9, 'nearest').unique().first().alias('90%'),
#     pl.col('delta_t').quantile(0.95, 'nearest').unique().first().alias('95%'),
#     pl.col('delta_t').quantile(0.99, 'nearest').unique().first().alias('99%')
# ])
print(marked_df.select(pl.col("delta_t").quantile(0.9, "nearest")))
print(marked_df.select(pl.col("delta_t").quantile(0.95, "nearest")))
print(marked_df.select(pl.col("delta_t").quantile(0.99, "nearest")))

shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ delta_t   │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 219501.0  │
│ null_count ┆ 0.0       │
│ mean       ┆ 31.009276 │
│ std        ┆ 44.443687 │
│ min        ┆ 1.0       │
│ 25%        ┆ 1.0       │
│ 50%        ┆ 4.0       │
│ 75%        ┆ 49.0      │
│ max        ┆ 313.0     │
└────────────┴───────────┘
shape: (1, 1)
┌─────────┐
│ delta_t │
│ ---     │
│ f64     │
╞═════════╡
│ 101.0   │
└─────────┘
shape: (1, 1)
┌─────────┐
│ delta_t │
│ ---     │
│ f64     │
╞═════════╡
│ 155.0   │
└─────────┘
shape: (1, 1)
┌─────────┐
│ delta_t │
│ ---     │
│ f64     │
╞═════════╡
│ 158.0   │
└─────────┘


In [7]:
# 현재 모델이 dt를 잘하는지 보려면, 간단한 상수 예측 baseline과 비교해보는 게 가장 빠릅니다.
import numpy as np
dt = marked_df["delta_t"].to_numpy()

for c in [1, 3, 10, 28, 47]:
    mae = np.mean(np.abs(dt - c))
    print(c, mae)

# 1 27.21605431309904
# 3 26.89743832548335
# 10 28.25687147985903
# 28 33.08285711933072
# 47 38.94298195052864

1 30.009275584165902
3 29.443619846834412
10 30.202071972337254
28 33.62441628967522
47 38.08453720028611


In [8]:
def build_rep_qty_table(marked_df: pl.DataFrame, K: int, tail_bins = (None, None)) -> pl.DataFrame:
    '''
    :param marked_df:
           Schema([('oper_part_no', String),
                   ('demand_dt', Int64),
                   ('seq', UInt32),
                   ('delta_t', Int32),
                   ('demand_qty', Float64),
                   ('z', Float64),
                   ('mark', Int32)])
    :param K: 8
    :param tail_bins: (K-2, K-1) 같은 tail bin id를 넣어주면 해당 bin은 p90으로 대체
    :return: DataFrame[mark: int, rep_qty: float]
    '''

    base = (
        marked_df
          .group_by('mark')
          .agg([
            pl.col('demand_qty').median().alias('rep_qty_median'),
            pl.col('demand_qty').quantile(0.90, 'nearest').alias('rep_qty_p90'),
            pl.len().alias('cnt'),
          ])
          .sort('mark')
    )

    b1, b2 = tail_bins
    if b1 is None: b1 = K - 2
    if b2 is None: b2 = K - 1

    # tail bin은 p90을 사용, 나머지는 median
    rep = (
        base
          .with_columns(
            pl.when(pl.col('mark').is_in([b1, b2]))
              .then(pl.col('rep_qty_p90'))
              .otherwise(pl.col('rep_qty_median'))
              .alias('rep_qty')
          )
          .select(['mark', 'rep_qty', 'cnt'])
    )

    return rep
import numpy as np
def rep_qty_to_tensor(rep_table: pl.DataFrame, K: int):
    rep = np.zeros((K,), dtype=np.float32)
    rows = rep_table.select(['mark', 'rep_qty']).to_numpy()
    for m, q in rows:
        rep[int(m)] = float(q)
    return rep

# K = 8
# rep_table = build_rep_qty_table(marked_df, K=K, tail_bins=(K-2, K-1))
# rep_qty = rep_qty_to_tensor(rep_table, K=K)  # shape (K,)
# rep_table

In [9]:
from utils.bptt_utils import predict_next
import numpy as np
import torch


@torch.no_grad()
def simulate_horizon_grid_with_user_wrapper(
    model,
    dt_hist: torch.Tensor,     # (L,) float
    mk_hist: torch.Tensor,     # (L,) long
    H: int,
    rep_qty: np.ndarray,       # (K_real,) float32  (PAD 제외)
    n_sims: int = 200,
    max_events: int = 200,
    use_sampling: bool = True,
    min_dt: float = 1.0,
    device: str = "cpu",
    week_index_rule: str = "ceil_minus_1",  # "ceil_minus_1" | "floor"
    pad_id: int | None = None,              # None이면 model(=wrapper/core)에서 자동 추론
):
    """
    Horizon 시뮬레이션 + 주차 그리드 복원 (PAD-aware).

    Assumption: model.predict_next(marks, dts) -> (y_hat, dt_hat, prob)
      - marks: [1, L] long
      - dts:   [1, L] float
      - y_hat: [1] long (argmax over prob)  ※ wrapper가 argmax를 반환해도 무방
      - dt_hat:[1] float (sample)
      - prob:  [1, K_model] float (softmax already applied)
          where K_model can be K_real or (K_real + 1 with PAD)

    rep_qty:
      - shape (K_real,) : PAD 제외한 실제 mark들의 대표 수량
      - PAD가 샘플링되지 않도록 내부에서 확률 마스킹 처리함

    Returns:
      mean_grid: (H,) float32
      all_grids: (n_sims, H) float32
    """

    if week_index_rule not in ("ceil_minus_1", "floor"):
        raise ValueError("week_index_rule must be 'ceil_minus_1' or 'floor'")

    rep_qty_t = torch.tensor(rep_qty, dtype=torch.float32, device=device)

    dt_hist = dt_hist.to(device).float()
    mk_hist = mk_hist.to(device).long()

    all_grids = torch.zeros((n_sims, H), dtype=torch.float32, device=device)

    # pad_id 자동 추론: wrapper가 core model을 `model` 속성으로 들고 있는 경우를 우선 지원
    if pad_id is None:
        core = getattr(model, "model", None)  # 사용자 wrapper: self.model = core_model
        if core is not None and hasattr(core, "cfg") and hasattr(core.cfg, "num_marks"):
            pad_id = int(core.cfg.num_marks - 1)  # PAD를 마지막 id로 쓴다는 가정
        else:
            pad_id = None  # PAD 없다고 가정

    for s in range(n_sims):
        dt_seq = dt_hist.clone()
        mk_seq = mk_hist.clone()

        t = 0.0
        grid = torch.zeros((H,), dtype=torch.float32, device=device)

        for _ in range(max_events):
            # ---- next dt / mark distribution ----
            y_hat, dt_hat, prob = model.predict_next(
                    mk_seq.unsqueeze(0),
                    dt_seq.unsqueeze(0),
            )

            dt_next = float(dt_hat.squeeze().item())
            if dt_next < min_dt:
                dt_next = min_dt

            t = t + dt_next
            if t > H:
                break

            # ---- mark 선택 (PAD 제거) ----
            probs = prob[0].clone()  # (K_model,)

            if pad_id is not None and 0 <= pad_id < probs.numel():
                probs[pad_id] = 0.0  # PAD 확률 제거
                probs = probs / probs.sum().clamp_min(1e-12)

            if use_sampling:
                mk_next = int(torch.multinomial(probs, 1).item())
            else:
                mk_next = int(torch.argmax(probs).item())

            # mk_next가 rep_qty 범위를 벗어나면 (예: PAD가 들어왔거나 K 불일치)
            if mk_next < 0 or mk_next >= rep_qty_t.numel():
                # 안전장치: 해당 이벤트는 무시하고 history만 업데이트하지 않음(혹은 break)
                # 보통은 PAD 마스킹이 되어 여기로 오면 K 불일치 가능성이 큼
                continue

            # ---- time -> weekly grid ----
            if week_index_rule == "ceil_minus_1":
                w = int(np.ceil(t) - 1)
            else:
                w = int(np.floor(t))

            if 0 <= w < H:
                grid[w] += rep_qty_t[mk_next]

            # ---- 4) roll history ----
            dt_seq = torch.cat([dt_seq[1:], torch.tensor([dt_next], device=device)], dim=0)
            mk_seq = torch.cat([mk_seq[1:], torch.tensor([mk_next], device=device)], dim=0)

        all_grids[s] = grid

    mean_grid = all_grids.mean(dim=0)
    return mean_grid.detach().cpu().numpy(), all_grids.detach().cpu().numpy()


def grid_to_int_list(mean_grid: np.ndarray, rounding: str = "round") -> list[int]:
    """
    rounding: "round" | "floor" | "ceil"
    """
    if rounding == "round":
        return np.rint(mean_grid).astype(int).tolist()
    if rounding == "floor":
        return np.floor(mean_grid).astype(int).tolist()
    if rounding == "ceil":
        return np.ceil(mean_grid).astype(int).tolist()
    raise ValueError("rounding must be 'round', 'floor', or 'ceil'")

In [14]:
from utils.rntpp_wrapper import RMTPPWrapper

# rep_qty 준비
rep_table = build_rep_qty_table(marked_df, K=K, tail_bins=(K-2, K-1))
rep_qty = rep_qty_to_tensor(rep_table, K=K)

# 특정 자재 최근 L개 이벤트
L = 30
part = "ZZ90239"

part_ev = (
    marked_df
    .filter(pl.col("oper_part_no") == part)
    .sort("seq")
)

# 이벤트 수가 부족하면 L을 줄여야 함
dt_hist = torch.tensor(part_ev.tail(L)["delta_t"].to_numpy(), dtype=torch.float32)
mk_hist = torch.tensor(part_ev.tail(L)["mark"].to_numpy(), dtype=torch.long)

# 3) 예측
H = 104
wrapped = RMTPPWrapper(model)

mean_grid, all_grids = simulate_horizon_grid_with_user_wrapper(
    model=wrapped,
    dt_hist=dt_hist,
    mk_hist=mk_hist,
    H=H,
    rep_qty=rep_qty,
    n_sims=200,
    use_sampling=True,
    device="cuda",
    week_index_rule="ceil_minus_1",
)

mean_grid  # (13,) float
pred_int = grid_to_int_list(mean_grid, rounding="round")
pred_int

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [15]:
mean_grid

array([0.74      , 0.65      , 0.91999996, 0.7       , 0.62      ,
       0.78499997, 0.65999997, 0.5       , 0.355     , 0.29      ,
       0.22      , 0.315     , 0.225     , 0.285     , 0.21499999,
       0.11499999, 0.14      , 0.21499999, 0.185     , 0.265     ,
       0.16      , 0.14999999, 0.11      , 0.145     , 0.185     ,
       0.02      , 0.035     , 0.14      , 0.11      , 0.05      ,
       0.07      , 0.05      , 0.02      , 0.045     , 0.07      ,
       0.065     , 0.085     , 0.16499999, 0.12      , 0.185     ,
       0.14      , 0.105     , 0.16      , 0.19      , 0.125     ,
       0.16499999, 0.14      , 0.22      , 0.175     , 0.17      ,
       0.24499999, 0.22999999, 0.195     , 0.19      , 0.24      ,
       0.19      , 0.11      , 0.105     , 0.34      , 0.19999999,
       0.14999999, 0.29      , 0.175     , 0.31      , 0.225     ,
       0.13499999, 0.16      , 0.155     , 0.16      , 0.11      ,
       1.9799999 , 0.175     , 0.17      , 0.17999999, 0.26   